<a href="https://colab.research.google.com/github/flexfengfeng/SMU-Hackathon-Prep/blob/main/track1_emerging_tech_accountscout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📡 Track 1 · Emerging Technology
## AccountScout: An AI Sales-Intelligence Agent for Enterprise Account Managers

**Coaching session · ~60 min**

### The sponsor's challenge
**Singtel** (Singapore Enterprise, with its AI & Data Analytics unit) asks:

> *"How might telcos use AI and emerging technologies to help enterprise sales teams better understand their customers, identify their needs and opportunities, and recommend the right next action — enabling them to deliver a more relevant customer experience and unlock new growth?"*

### The real-world problem
An enterprise account manager at a Singapore telco has two jobs:
1. **Farming:** grow existing multinational customers (upsell, cross-sell, renewals).
2. **Hunting:** win new customers overseas, e.g. in India, where the telco has few relationships and little local knowledge.

The information is scattered across the CRM, support tickets, news, industry reports and regulation updates. Joining it up is manual and slow, so engagement is **reactive**: the team responds to a request for quote instead of spotting the need first.

### The emerging-tech angle
- **Structured output** turns messy evidence into a typed account brief your code can check.
- **AI agents** (LLMs with **function calling**) can explore an unfamiliar market by calling your data tools.
- **Python, not the LLM, does the scoring**, so every number is explainable to a sales leader.

### What you will build
| Step | Technique | What it does |
|---|---|---|
| Data | Mock CRM, news signals, catalogue, prospects | Fictional stand-ins for the systems a sales team already has |
| Scoring | **Plain Python** | Transparent 0–100 opportunity score: signal recency, fit, account size, renewal window |
| Farm | **Structured output** + ID checks | Existing customer ➜ validated account brief with cited evidence |
| Hunt | **Function calling / agent** | The model explores prospects in India with your tools and ranks them |
| Guardrail | **Human-in-the-loop** | The account manager edits and approves before anything is logged; nothing is sent |
| Measure | **Pipeline dashboard** | Qualified opportunities, conversion proxy, share-of-wallet growth |
| Demo | **Gradio** | Pick Farm or Hunt ➜ brief, ranked opportunities, draft outreach, Approve button |

## Step 1: Install Dependencies
- `google-genai`: Google's Gen AI SDK for Gemini (already preinstalled in Colab).
- `gradio`: the web UI for the demo.

`pydantic`, `pandas` and `matplotlib` are already in Colab.

The first line records the versions of a few core packages Colab depends on, so the install below doesn't upgrade them and break Colab. If Colab asks you to **restart the session** after installing, click restart and continue from Step 2.

In [ ]:
# Keep Colab's own versions of these packages to avoid dependency conflicts
!pip freeze | grep -iE '^(google-auth|opentelemetry-(api|sdk))==' > /tmp/colab-constraints.txt
!pip install -q -c /tmp/colab-constraints.txt google-genai gradio==6.29.1

## Step 2: Setup Google Gemini API Key
1. Go to [Google AI Studio](https://aistudio.google.com/).
2. Sign in with your Google account and click **Get API key**.
3. In Colab, click the 🔑 **Secrets** icon on the left sidebar.
4. Create a new secret named `GEMINI_API_KEY` and paste your key.
5. Turn on the **Notebook access** toggle for that secret.

We also set the client to wait and retry when the free tier says *"429: too many requests"*. Free-tier keys only allow about 5 requests per minute, so every team will hit this during the hackathon. If a cell seems to pause for up to a minute, it is waiting out the limit, not stuck.

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

# Free-tier keys only allow a few requests per minute. These options make the SDK
# wait and retry any single request that gets "429: too many requests".
# Retrying per request matters for the agent in Step 7: one agent run makes several
# requests (one per tool call), and we must not restart the whole run halfway through.
RETRY = types.HttpRetryOptions(attempts=6, initial_delay=15, max_delay=60, http_status_codes=[429, 500, 503])

# Securely load the API key from Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY, http_options=types.HttpOptions(retry_options=RETRY))
    print("✅ Gemini API Key configured successfully!")
except Exception as e:
    print("❌ Error: Please ensure you added 'GEMINI_API_KEY' to your Colab Secrets and turned on 'Notebook access'.")
    print("   Details:", e)
    raise

# One place to change the model for the whole notebook
MODEL = "gemini-3.5-flash-lite"

def ask_gemini(contents, config=None):
    """Call Gemini. Rate-limit retries are handled by the client (see RETRY above)."""
    return client.models.generate_content(model=MODEL, contents=contents, config=config)

## Step 3: The Sales Team's Scattered Data (Mock Data)
A real version would read the telco's CRM (e.g. Salesforce), its support-ticket system, a news feed and the product catalogue. Here we fake all four with Python lists for a fictional **Singapore telco**:

- `CRM_ACCOUNTS`: existing multinational customers, with products held, contract end date, spend, support tickets and last contact.
- `SIGNALS`: dated news items with a source. Some are about one company, some about a whole industry or market (e.g. India's DPDP Act).
- `CATALOGUE`: what the telco can sell, each with an ID like `SOL-SDWAN`.
- `PROSPECTS`: companies in India (and one in Vietnam) the telco doesn't serve yet. **Public-style information only, and job roles instead of people.**

> ⚠️ Every company, headline and number below is **fictional**. The brief comes from Singtel, but this is not Singtel's data.

> 💡 **Hackathon tip:** every record has an **ID**. IDs are what let us check later that the AI only cites things that really exist.

In [ ]:
import pandas as pd
from datetime import date

TODAY = date(2026, 11, 4)  # fixed so everyone gets the same scores

CATALOGUE = {
    "SOL-SDWAN": {"name": "Managed SD-WAN", "annual_value_sgd": 180_000, "buyer_role": "Head of IT Infrastructure",
                  "tags": ["new-site", "multi-site", "connectivity", "latency", "expansion"]},
    "SOL-IPLC":  {"name": "International private leased circuit (SG-India)", "annual_value_sgd": 240_000, "buyer_role": "CIO",
                  "tags": ["india", "cross-border", "expansion", "data-centre"]},
    "SOL-5GPN":  {"name": "5G private network", "annual_value_sgd": 350_000, "buyer_role": "COO / Head of Operations",
                  "tags": ["factory", "automation", "port", "new-site"]},
    "SOL-CLOUD": {"name": "Managed multi-cloud", "annual_value_sgd": 300_000, "buyer_role": "CIO",
                  "tags": ["cloud", "migration", "data-residency", "leadership-change"]},
    "SOL-MSS":   {"name": "Managed security (SOC + compliance)", "annual_value_sgd": 220_000, "buyer_role": "CISO",
                  "tags": ["security", "ransomware", "compliance", "dpdp"]},
    "SOL-IOT":   {"name": "IoT fleet and asset tracking", "annual_value_sgd": 150_000, "buyer_role": "Head of Supply Chain",
                  "tags": ["fleet", "cold-chain", "tracking", "warehouse"]},
    "SOL-MOBILE": {"name": "Enterprise mobile plans", "annual_value_sgd": 60_000, "buyer_role": "Head of IT",
                   "tags": ["headcount", "new-site"]},
}

# Existing customers. Spend and wallet (estimated total ICT spend) are per year, in SGD.
CRM_ACCOUNTS = [
    {"id": "ACC-001", "company": "Kestrel Precision Components", "industry": "manufacturing",
     "markets": ["Singapore", "Malaysia", "India"], "products_held": ["SOL-MOBILE", "SOL-CLOUD"],
     "contract_end": "2027-03-31", "annual_spend_sgd": 1_100_000, "ict_wallet_sgd": 5_200_000, "last_contact": "2026-06-12",
     "tickets": [{"id": "TCK-881", "date": "2026-09-14", "summary": "Intermittent latency on SG-Penang VPN"},
                 {"id": "TCK-902", "date": "2026-10-20", "summary": "Asked for a quote: 40 extra mobile lines for a new team"}]},
    {"id": "ACC-002", "company": "Lumen Harbour Logistics", "industry": "logistics",
     "markets": ["Singapore", "Indonesia", "India"], "products_held": ["SOL-SDWAN", "SOL-MOBILE"],
     "contract_end": "2027-10-31", "annual_spend_sgd": 700_000, "ict_wallet_sgd": 2_800_000, "last_contact": "2026-09-30",
     "tickets": [{"id": "TCK-870", "date": "2026-08-02", "summary": "GPS trackers from another vendor dropping offline"}]},
    {"id": "ACC-003", "company": "Orchid Biopharma Asia", "industry": "pharma",
     "markets": ["Singapore", "India"], "products_held": ["SOL-IPLC", "SOL-MSS"],
     "contract_end": "2028-04-30", "annual_spend_sgd": 2_300_000, "ict_wallet_sgd": 7_500_000, "last_contact": "2026-10-05",
     "tickets": []},
    {"id": "ACC-004", "company": "Tembusu Retail Group", "industry": "retail",
     "markets": ["Singapore", "Malaysia"], "products_held": ["SOL-MOBILE", "SOL-SDWAN", "SOL-CLOUD"],
     "contract_end": "2027-01-31", "annual_spend_sgd": 900_000, "ict_wallet_sgd": 2_000_000, "last_contact": "2026-02-18",
     "tickets": []},
]

# Companies we don't serve yet. Public-style facts only: no names, no contact details.
PROSPECTS = [
    {"id": "PRO-101", "company": "Nilgiri Auto Systems", "industry": "manufacturing", "market": "India", "markets": ["India"],
     "city": "Chennai", "employees": "2,000-5,000", "ict_wallet_sgd": 3_500_000, "singapore_link": "Plans a regional sales office in Singapore"},
    {"id": "PRO-102", "company": "Saffron Ledger Fintech", "industry": "fintech", "market": "India", "markets": ["India"],
     "city": "Bengaluru", "employees": "500-1,000", "ict_wallet_sgd": 2_500_000, "singapore_link": "Applying for a Singapore payments licence"},
    {"id": "PRO-103", "company": "Deccan Cold Chain", "industry": "logistics", "market": "India", "markets": ["India"],
     "city": "Hyderabad", "employees": "1,000-2,000", "ict_wallet_sgd": 1_800_000, "singapore_link": "Ships pharma exports via Singapore"},
    {"id": "PRO-104", "company": "Konkan Textiles", "industry": "manufacturing", "market": "India", "markets": ["India"],
     "city": "Surat", "employees": "1,000-2,000", "ict_wallet_sgd": 1_200_000, "singapore_link": "None known"},
    {"id": "PRO-105", "company": "Brahmaputra Agritech", "industry": "agritech", "market": "India", "markets": ["India"],
     "city": "Guwahati", "employees": "100-500", "ict_wallet_sgd": 600_000, "singapore_link": "None known"},
    {"id": "PRO-201", "company": "Mekong Fabrication", "industry": "manufacturing", "market": "Vietnam", "markets": ["Vietnam"],
     "city": "Hai Phong", "employees": "1,000-2,000", "ict_wallet_sgd": 2_000_000, "singapore_link": "Parent company listed in Singapore"},
]

# News and market signals. company=None means it applies to a whole industry or market.
SIGNALS = [
    {"id": "SIG-01", "date": "2026-10-08", "company": "Kestrel Precision Components", "industry": None, "market": None,
     "source": "Trade news (fictional)", "headline": "Kestrel Precision Components to open new plant in Pune by mid-2027",
     "tags": ["india", "new-site", "expansion", "factory", "cross-border"]},
    {"id": "SIG-02", "date": "2026-09-20", "company": "Kestrel Precision Components", "industry": None, "market": None,
     "source": "Company careers page (fictional)", "headline": "Kestrel hiring 40 automation engineers for Pune",
     "tags": ["automation", "headcount"]},
    {"id": "SIG-03", "date": "2026-10-22", "company": None, "industry": "manufacturing", "market": None,
     "source": "Regional cyber-security news (fictional)", "headline": "Ransomware halts production at a Penang electronics maker for five days",
     "tags": ["security", "ransomware"]},
    {"id": "SIG-04", "date": "2026-08-30", "company": None, "industry": None, "market": "India",
     "source": "Regulator update (summary)", "headline": "India's DPDP Act rules phase in: firms prepare for data-protection compliance deadlines",
     "tags": ["compliance", "dpdp", "data-residency"]},
    {"id": "SIG-05", "date": "2026-10-15", "company": "Lumen Harbour Logistics", "industry": None, "market": None,
     "source": "Port authority press release (fictional)", "headline": "Lumen Harbour Logistics wins berth operations contract at Chennai port",
     "tags": ["india", "expansion", "port", "new-site", "cross-border"]},
    {"id": "SIG-06", "date": "2026-09-05", "company": "Lumen Harbour Logistics", "industry": None, "market": None,
     "source": "Company press release (fictional)", "headline": "Lumen launches temperature-controlled pharma shipping lane",
     "tags": ["cold-chain", "tracking", "fleet"]},
    {"id": "SIG-07", "date": "2026-10-01", "company": "Orchid Biopharma Asia", "industry": None, "market": None,
     "source": "Business news (fictional)", "headline": "Orchid Biopharma Asia appoints a new group CIO with a cloud-first track record",
     "tags": ["leadership-change", "cloud", "migration"]},
    {"id": "SIG-08", "date": "2026-07-18", "company": "Orchid Biopharma Asia", "industry": None, "market": None,
     "source": "Company annual report (fictional)", "headline": "Orchid expands Bengaluru R&D centre to 600 staff",
     "tags": ["india", "expansion", "headcount", "data-centre"]},
    {"id": "SIG-09", "date": "2026-03-02", "company": "Tembusu Retail Group", "industry": None, "market": None,
     "source": "Retail trade news (fictional)", "headline": "Tembusu Retail refreshes 12 stores in Kuala Lumpur",
     "tags": ["multi-site"]},
    {"id": "SIG-10", "date": "2026-10-12", "company": "Nilgiri Auto Systems", "industry": None, "market": None,
     "source": "Auto industry news (fictional)", "headline": "Nilgiri Auto Systems commissions a robotic assembly line in Chennai",
     "tags": ["automation", "factory", "new-site"]},
    {"id": "SIG-11", "date": "2026-09-28", "company": "Nilgiri Auto Systems", "industry": None, "market": None,
     "source": "Business news (fictional)", "headline": "Nilgiri plans Singapore regional office to serve ASEAN carmakers",
     "tags": ["cross-border", "expansion"]},
    {"id": "SIG-12", "date": "2026-10-18", "company": "Saffron Ledger Fintech", "industry": None, "market": None,
     "source": "Fintech news (fictional)", "headline": "Saffron Ledger Fintech applies for a Singapore payments licence",
     "tags": ["cross-border", "expansion", "compliance", "security"]},
    {"id": "SIG-13", "date": "2026-08-14", "company": "Saffron Ledger Fintech", "industry": None, "market": None,
     "source": "Company blog (fictional)", "headline": "Saffron Ledger moves its core platform to the cloud",
     "tags": ["cloud", "migration", "data-residency"]},
    {"id": "SIG-14", "date": "2026-09-10", "company": "Deccan Cold Chain", "industry": None, "market": None,
     "source": "Logistics news (fictional)", "headline": "Deccan Cold Chain adds 200 refrigerated trucks",
     "tags": ["fleet", "cold-chain", "tracking"]},
    {"id": "SIG-15", "date": "2026-02-20", "company": "Konkan Textiles", "industry": None, "market": None,
     "source": "Stock exchange filing (fictional)", "headline": "Konkan Textiles reports flat annual sales",
     "tags": []},
    {"id": "SIG-16", "date": "2026-10-05", "company": "Mekong Fabrication", "industry": None, "market": None,
     "source": "Vietnam industry news (fictional)", "headline": "Mekong Fabrication breaks ground on a second Hai Phong factory",
     "tags": ["new-site", "factory", "expansion"]},
    {"id": "SIG-17", "date": "2026-10-25", "company": None, "industry": "logistics", "market": None,
     "source": "Regional cyber-security news (fictional)", "headline": "South Asian port operators step up security after a phishing wave",
     "tags": ["security"]},
]

# What each industry typically cares about (used for "fit" in the score)
INDUSTRY_TAGS = {"manufacturing": ["factory", "automation"], "logistics": ["fleet", "warehouse", "port"],
                 "pharma": ["compliance", "cold-chain"], "fintech": ["security", "compliance", "cloud"],
                 "retail": ["multi-site"], "agritech": ["cold-chain", "tracking"]}

print(f"Loaded {len(CRM_ACCOUNTS)} customers, {len(PROSPECTS)} prospects, {len(SIGNALS)} signals, {len(CATALOGUE)} solutions.")
display(pd.DataFrame(CRM_ACCOUNTS)[["id", "company", "industry", "products_held", "contract_end", "annual_spend_sgd", "last_contact"]])

## Step 4: Score Opportunities in Plain Python
Before any AI, we decide **how an opportunity is scored**. A sales leader will ask *"why is this account at the top?"* and "the AI said so" is not an answer.

Each (company, solution) pair gets 0–100 points:
| Part | Max | Rule |
|---|---|---|
| Recency | 30 | Newest signal that matches the solution; fades to 0 after 180 days |
| Fit | 30 | How many of the solution's tags match the company's signals and industry (3 matches = full marks) |

News about the company itself counts fully; news about its whole industry or market (e.g. a regulation) counts half.
| Size | 20 | Estimated ICT wallet; S$6m or more = full marks |
| Renewal | 20 | Customers: contract ends within 6 months = 20, within 12 = 10, later = 3. Prospects: 8 (we can't see their contracts) |

Solutions the customer **already holds** are skipped. The LLM never invents a score: it reads these numbers.

In [ ]:
def find_company(name):
    """Look a company up in the CRM first, then the prospect list. Returns (kind, record)."""
    n = name.lower().strip()
    for kind, records in (("customer", CRM_ACCOUNTS), ("prospect", PROSPECTS)):
        for rec in records:
            if n and (n in rec["company"].lower() or n == rec["id"].lower()):
                return kind, rec
    return None, None

def signals_for(rec):
    """Company news + news about its industry + news about markets it operates in (e.g. regulation)."""
    return [s for s in SIGNALS if s["company"] == rec["company"]
            or (s["company"] is None and s["industry"] == rec["industry"])
            or (s["company"] is None and s["market"] in rec["markets"])]

def score_opp(company, solution_id):
    """Transparent opportunity score. Returns None if the company already has this solution."""
    kind, rec = find_company(company)
    sol = CATALOGUE[solution_id]
    if solution_id in rec.get("products_held", []):
        return None
    sigs = signals_for(rec)
    # Company-specific news counts fully; industry- or market-wide news counts half
    weight = lambda s: 1 if s["company"] == rec["company"] else 0.5
    own_tags = set(INDUSTRY_TAGS.get(rec["industry"], [])).union(*[s["tags"] for s in sigs if weight(s) == 1])
    wide_tags = set().union(*[s["tags"] for s in sigs if weight(s) < 1]) - own_tags
    relevant = [s for s in sigs if set(s["tags"]) & set(sol["tags"])]
    recency = 30 * max([weight(s) * max(0, 1 - (TODAY - date.fromisoformat(s["date"])).days / 180) for s in relevant], default=0)
    fit = 30 * min(1, (len(own_tags & set(sol["tags"])) + 0.5 * len(wide_tags & set(sol["tags"]))) / 3)
    size = 20 * min(1, rec["ict_wallet_sgd"] / 6_000_000)
    if kind == "customer":
        months_left = (date.fromisoformat(rec["contract_end"]) - TODAY).days / 30.4
        renewal = 20 if months_left <= 6 else 10 if months_left <= 12 else 3
    else:
        renewal = 8
    return {"company": rec["company"], "solution_id": solution_id, "solution": sol["name"],
            "score": round(recency + fit + size + renewal), "recency": round(recency), "fit": round(fit),
            "size": round(size), "renewal": renewal, "signal_ids": [s["id"] for s in relevant],
            "est_annual_value_sgd": sol["annual_value_sgd"], "buyer_role": sol["buyer_role"]}

def rank_opportunities(company):
    """All solutions this company doesn't have yet, best score first."""
    rows = [score_opp(company, sid) for sid in CATALOGUE]
    return pd.DataFrame([r for r in rows if r]).sort_values("score", ascending=False).reset_index(drop=True)

def rank_prospects(market, industry=""):
    """Best opportunity per prospect in a market, best first."""
    best = [rank_opportunities(p["company"]).iloc[0] for p in PROSPECTS
            if p["market"].lower() == market.lower() and industry.lower() in p["industry"]]
    return pd.DataFrame(best).sort_values("score", ascending=False).reset_index(drop=True) if best else pd.DataFrame()

FARM_ACCOUNT = "Kestrel Precision Components"
display(rank_opportunities(FARM_ACCOUNT)[["solution_id", "solution", "score", "recency", "fit", "size", "renewal", "signal_ids"]])

Kestrel's contract ends in under 5 months, it is opening a plant in Pune, and a peer manufacturer was just hit by ransomware. The table turns that into a ranked list in milliseconds, with **no LLM call**.

> ✍️ The weights are a starting guess. In a pilot you would tune them on past wins and losses. Being able to say *"we can tune this"* is a strong answer for judges.

## Step 5: Farming: An Account Brief with Structured Output
For a customer we **already serve**, we know exactly what to look up: their CRM record, their signals, and the top-scored solutions. So plain Python gathers the evidence, and **one** LLM call turns it into an account brief.

The brief is a **Pydantic schema**, so Gemini must return JSON that fits it and the SDK hands us a Python object (`response.parsed`). Every buying signal and solution in it carries an **ID**.

Then comes the anti-hallucination step: `check_brief` checks **in Python** that every cited ID really exists in the evidence, removes any that don't, and flags anything that looks like an invented person's name.

> 💡 Why not an agent here? When the path is known, a fixed workflow is cheaper (1 request), faster and more predictable. Save the agent for when the path is *not* known (Step 7).

In [ ]:
import json, re
from typing import Literal
from pydantic import BaseModel, Field
from IPython.display import Markdown

class BuyingSignal(BaseModel):
    signal_id: str = Field(description="ID of a signal from the evidence, e.g. SIG-01")
    why_it_matters: str = Field(description="One sentence: what this signal means for the telco")

class RecommendedSolution(BaseModel):
    solution_id: str = Field(description="ID from the candidate solutions, e.g. SOL-SDWAN")
    rationale: str = Field(description="One sentence linking the solution to a need")

class AccountBrief(BaseModel):
    company: str
    summary: str = Field(description="2-3 sentences: what is changing at this company and why it matters to us")
    buying_signals: list[BuyingSignal]
    likely_needs: list[str] = Field(description="2-4 short business needs")
    recommended_solutions: list[RecommendedSolution] = Field(description="1-3 solutions, best first")
    decision_maker_role: str = Field(description="A job title only, e.g. 'CIO'. Never a person's name.")
    next_best_action: str = Field(description="One concrete action for the account manager")
    timing: Literal["this week", "this month", "this quarter", "next quarter"]
    confidence: Literal["low", "medium", "high"]
    evidence_ids: list[str] = Field(description="Every signal, ticket or account ID you relied on")
    outreach_subject: str
    outreach_body: str = Field(description="Email under 120 words, addressed to the role, no names, signed 'Your account team'")

BRIEF_PROMPT = """You are AccountScout, helping an enterprise account manager at a Singapore telco.
Write an account brief using ONLY the evidence below.
- Cite signal IDs exactly as given. Recommend only solution IDs from the candidate list (already ranked by our scoring model).
- Suggest a decision-maker ROLE, never a person's name. Do not invent facts, people or numbers.
- {mode_hint}

Evidence:
"""
MODE_HINTS = {
    "farm": "This is an existing customer: look for upsell, cross-sell and renewal timing, and use open support tickets if relevant.",
    "hunt": "This is a new prospect in a market where we are not well known: lead with their priorities and suggest a low-commitment first step.",
}

def gather_evidence(company):
    """Everything the LLM is allowed to use, collected by Python."""
    kind, rec = find_company(company)
    candidates = rank_opportunities(rec["company"]).head(4)
    return {"kind": kind, "profile": rec, "signals": signals_for(rec),
            "candidates": candidates[["solution_id", "solution", "score", "signal_ids", "buyer_role"]].to_dict("records")}

NAME_PATTERN = re.compile(r"\b(Mr|Ms|Mrs|Mdm|Dr)\.?\s+[A-Z][a-z]+")

def check_brief(brief, ev):
    """Anti-hallucination: keep only IDs that exist in the evidence. Returns (brief, problems)."""
    problems = []
    signal_ids = {s["id"] for s in ev["signals"]}
    held = set(ev["profile"].get("products_held", []))
    known_ids = signal_ids | {ev["profile"]["id"]} | {t["id"] for t in ev["profile"].get("tickets", [])}

    problems += [f"Removed unknown signal {s.signal_id}" for s in brief.buying_signals if s.signal_id not in signal_ids]
    brief.buying_signals = [s for s in brief.buying_signals if s.signal_id in signal_ids]
    problems += [f"Removed {r.solution_id}: not in catalogue or already held" for r in brief.recommended_solutions
                 if r.solution_id not in CATALOGUE or r.solution_id in held]
    brief.recommended_solutions = [r for r in brief.recommended_solutions if r.solution_id in CATALOGUE and r.solution_id not in held]
    problems += [f"Removed unknown evidence ID {e}" for e in brief.evidence_ids if e not in known_ids]
    brief.evidence_ids = [e for e in brief.evidence_ids if e in known_ids]

    if NAME_PATTERN.search(brief.decision_maker_role + " " + brief.outreach_body):
        problems.append("Possible person's name in the role or email: replace it with a job title")
    if not brief.buying_signals or not brief.recommended_solutions:
        brief.confidence = "low"
        problems.append("No valid evidence left, so confidence was set to low")
    return brief, problems

def make_brief(company):
    """1 LLM call: evidence in, checked AccountBrief out."""
    ev = gather_evidence(company)
    mode = "farm" if ev["kind"] == "customer" else "hunt"
    prompt = BRIEF_PROMPT.format(mode_hint=MODE_HINTS[mode]) + json.dumps(ev, indent=1, default=str)
    response = ask_gemini(prompt, types.GenerateContentConfig(response_mime_type="application/json", response_schema=AccountBrief))
    brief, problems = check_brief(response.parsed, ev)
    return brief, problems, ev

def brief_markdown(brief, ev):
    headlines = {s["id"]: s for s in SIGNALS}
    lines = [f"### {ev['profile']['company']} ({'Farm: existing customer' if ev['kind'] == 'customer' else 'Hunt: new prospect'})",
             brief.summary, "", "**Buying signals**"]
    lines += [f"- `{s.signal_id}` {headlines[s.signal_id]['headline']} *({headlines[s.signal_id]['source']}, "
              f"{headlines[s.signal_id]['date']})*: {s.why_it_matters}" for s in brief.buying_signals]
    lines += ["", "**Likely needs:** " + "; ".join(brief.likely_needs), "", "**Recommended solutions**"]
    lines += [f"- `{r.solution_id}` {CATALOGUE[r.solution_id]['name']}: {r.rationale}" for r in brief.recommended_solutions]
    lines += ["", f"**Talk to (role):** {brief.decision_maker_role}  ", f"**Next best action:** {brief.next_best_action}  ",
              f"**Timing:** {brief.timing} · **Confidence:** {brief.confidence}  ", f"**Evidence:** {', '.join(brief.evidence_ids)}"]
    return "\n".join(lines)

farm_brief, farm_problems, farm_ev = make_brief(FARM_ACCOUNT)
display(Markdown(brief_markdown(farm_brief, farm_ev)))
print("🔎 ID check:", farm_problems or "all cited IDs exist ✅")

## Step 6: Give the Hunting Agent Tools (Function Calling)
Hunting in India is different. The account manager doesn't know which companies matter, so the path isn't fixed: look at the prospects, read their signals, score the promising ones, maybe look up a solution for an unusual need. That's what an **agent** is for.

We write normal Python functions and pass them to Gemini. The SDK reads each function's **name, type hints and docstring** and describes them to the model. The model decides which tool to call and with what arguments; the SDK runs it and sends the result back, looping until the model has its answer (**automatic function calling**).

Two design choices to notice:
- `score_opportunity` just calls the Step 4 formula, so the agent **can't make up scores**.
- There is **no tool to log or send anything**. The agent can research and recommend; only a human can act (Step 8). This is called *least privilege*.

> ✍️ Docstrings matter: they are the model's only documentation for your tools.

In [ ]:
TOOL_LOG = []

def _log(tool, args, result):
    TOOL_LOG.append({"tool": tool, "args": args, "result": result})
    return result

def search_crm(company: str) -> dict:
    """Check whether a company is already our customer, and get its CRM record if so.

    Args:
        company: company name or account ID, e.g. "Kestrel" or "ACC-001".
    """
    kind, rec = find_company(company)
    return _log("search_crm", {"company": company}, {"account": rec if kind == "customer" else "not a customer"})

def get_signals(company: str) -> dict:
    """Get dated news and market signals (with signal IDs and sources) relevant to a company.

    Args:
        company: exact company name as returned by get_prospects or search_crm.
    """
    kind, rec = find_company(company)
    sigs = [{k: s[k] for k in ("id", "date", "source", "headline")} for s in signals_for(rec)] if rec else []
    return _log("get_signals", {"company": company}, {"signals": sigs})

def match_solutions(need: str) -> dict:
    """Find catalogue solutions for a business need, e.g. "connect a new factory" or "data protection compliance".

    Args:
        need: the need in plain words.
    """
    words = set(need.lower().replace("-", " ").split())
    hits = {sid: s["name"] for sid, s in CATALOGUE.items()
            if words & set(" ".join(s["tags"] + [s["name"].lower()]).replace("-", " ").split())}
    return _log("match_solutions", {"need": need}, {"solutions": hits or "no match"})

def get_prospects(market: str, industry: str) -> dict:
    """List companies in a market that are not yet our customers. Public information only, no personal data.

    Args:
        market: country, e.g. "India" or "Vietnam".
        industry: e.g. "manufacturing", or "" for all industries.
    """
    found = [{k: p[k] for k in ("id", "company", "industry", "city", "employees", "singapore_link")} for p in PROSPECTS
             if p["market"].lower() == market.lower() and industry.lower() in p["industry"]]
    return _log("get_prospects", {"market": market, "industry": industry}, {"prospects": found})

def score_opportunity(company: str, solution_id: str) -> dict:
    """Score an opportunity 0-100 with our transparent formula (recency, fit, size, renewal).

    Args:
        company: exact company name.
        solution_id: a catalogue ID such as "SOL-SDWAN", or "" to get the two best-fit solutions.
    """
    if not find_company(company)[1]:
        return _log("score_opportunity", {"company": company}, {"error": "unknown company"})
    if solution_id:
        options = [score_opp(company, solution_id)] if solution_id in CATALOGUE else []
    else:
        options = rank_opportunities(company).head(2).to_dict("records")
    options = [{k: o[k] for k in ("company", "solution_id", "score", "signal_ids")} for o in options if o]
    return _log("score_opportunity", {"company": company, "solution_id": solution_id}, {"options": options})

TOOLS = [search_crm, get_signals, match_solutions, get_prospects, score_opportunity]
print("🧰 Tools ready:", [t.__name__ for t in TOOLS])

## Step 7: Run the Agent: Hunting in India
The **system instruction** is the agent's job description: a clear order of work, and what *not* to do.

⏱️ **Rate limits:** every tool round trip is one more request. We ask the model to make independent calls **in the same turn** (all `get_signals` calls together), which keeps a run to about **4 requests**. `maximum_remote_calls` is a hard cap, so a confused agent can't burn your quota.

After the agent finishes, we read the winning prospect from the **tool log** (Python numbers), not from the model's prose, and reuse `make_brief` from Step 5 for it.

In [ ]:
HUNT_INSTRUCTIONS = """You are AccountScout, a sales-research agent for a Singapore telco's enterprise team.
Your task is to HUNT for new business in the market you are given. Work in this order:
1. Call get_prospects once.
2. Call get_signals for every prospect, all in the same turn.
3. Call score_opportunity (solution_id "") for every prospect with at least one company-specific signal, all in the same turn.
4. Use match_solutions only if a signal suggests a need you cannot place.
Rules: only use company names, signal IDs and solution IDs returned by the tools. Never invent people or contact details;
suggest job roles instead. Scores come only from score_opportunity.
Finish with a ranked shortlist of the top 3 (max 150 words): company, score, best solution ID, the signal IDs behind it,
and one suggested first action."""

def run_hunt_agent(market, industry=""):
    TOOL_LOG.clear()
    config = types.GenerateContentConfig(
        system_instruction=HUNT_INSTRUCTIONS, tools=TOOLS,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=6),
    )
    response = ask_gemini(f"Market: {market}. Industry focus: {industry or 'any'}.", config)
    return response.text or "(The agent hit the request cap before writing a summary.)"

def top_from_log():
    """The highest Python score the agent saw, read from the tool log."""
    options = [o for c in TOOL_LOG if c["tool"] == "score_opportunity" for o in c["result"].get("options", [])]
    return max(options, key=lambda o: o["score"])["company"] if options else None

HUNT_MARKET = "India"
hunt_summary = run_hunt_agent(HUNT_MARKET)

print("🔧 Agent's tool calls:")
for i, call in enumerate(TOOL_LOG, 1):
    print(f"  {i}. {call['tool']}({call['args']})")
print("\n📋 Agent's shortlist:\n", hunt_summary)

top_prospect = top_from_log() or rank_prospects(HUNT_MARKET).iloc[0]["company"]
hunt_brief, hunt_problems, hunt_ev = make_brief(top_prospect)
display(Markdown(brief_markdown(hunt_brief, hunt_ev)))
print("🔎 ID check:", hunt_problems or "all cited IDs exist ✅")

### 🤔 Coaching discussion (5 min)
- Look at the tool-call trail and the scores. Brahmaputra Agritech ranks last mainly because **nobody writes news about it**. Is that a real lack of need, or a bias towards big, newsworthy companies? (Smaller firms in less-covered regions may get missed.)
- Compare Farm (fixed workflow, 1 request) with Hunt (agent, ~4 requests + 1 for the brief). When is the extra cost of an agent worth it?
- Kestrel's brief may mention the **ransomware incident at a peer**. That's public news. What if it had come from *another customer's* support tickets? (Hint: client confidentiality across accounts.)
- The brief gives a **role**, not a name. How would a real account manager find the right person, legally and respectfully?

## Step 8: Human in the Loop: Approve Before Anything Is Logged
AccountScout **never sends anything**. It drafts; the account manager edits and approves; only then does `log_opportunity` write a row to the pipeline (a stand-in for creating an opportunity in the CRM).

Note that `log_opportunity` is **not** in the agent's `TOOLS`. Python also re-scores the opportunity itself, so the logged numbers can't be changed by the model.

In [ ]:
PIPELINE = []
QUALIFIED_SCORE = 60  # opportunities at or above this score count as "qualified"

def log_opportunity(brief, ev, subject, body, approved_by, approved):
    """Write an approved opportunity to the pipeline. Deliberately NOT given to the agent."""
    if not approved:
        return "⏸️ Not logged: the account manager has not approved it."
    if not brief.recommended_solutions:
        return "⚠️ Not logged: the brief has no valid solution."
    rec, farm = ev["profile"], ev["kind"] == "customer"
    s = score_opp(rec["company"], brief.recommended_solutions[0].solution_id)
    # Conversion proxy: higher score = more likely to win; existing relationships convert more often
    win_prob = round(0.05 + 0.45 * s["score"] / 100 + (0.10 if farm else 0), 2)
    spend = rec.get("annual_spend_sgd", 0)
    opp_id = f"OPP-{101 + len(PIPELINE)}"
    PIPELINE.append({
        "opp_id": opp_id, "mode": "Farm" if farm else "Hunt", "company": rec["company"], "solution_id": s["solution_id"],
        "score": s["score"], "qualified": s["score"] >= QUALIFIED_SCORE, "est_value_sgd": s["est_annual_value_sgd"],
        "win_prob": win_prob, "weighted_value_sgd": round(win_prob * s["est_annual_value_sgd"]),
        "sow_now": round(spend / rec["ict_wallet_sgd"], 3),
        "sow_if_won": round((spend + s["est_annual_value_sgd"]) / rec["ict_wallet_sgd"], 3),
        "next_action": brief.next_best_action, "timing": brief.timing, "evidence": ", ".join(brief.evidence_ids),
        "outreach_subject": subject, "outreach_body": body, "status": "Approved, NOT sent", "approved_by": approved_by,
    })
    return f"✅ Logged {opp_id}: {rec['company']} / {s['solution_id']} (score {s['score']}). Nothing was sent."

# 1) Farm: the account manager reads the draft and adds a personal line before approving
print("📧 Draft:", farm_brief.outreach_subject, "\n", farm_brief.outreach_body, "\n")
edited_body = farm_brief.outreach_body + "\n\nP.S. Happy to bring our India network specialist to the call."
print(log_opportunity(farm_brief, farm_ev, farm_brief.outreach_subject, edited_body, approved_by="Account manager (you)", approved=True))

# 2) Hunt: first not yet approved, then approved
print(log_opportunity(hunt_brief, hunt_ev, hunt_brief.outreach_subject, hunt_brief.outreach_body, "Account manager (you)", approved=False))
print(log_opportunity(hunt_brief, hunt_ev, hunt_brief.outreach_subject, hunt_brief.outreach_body, "Account manager (you)", approved=True))

## Step 9: The Pipeline Dashboard
Singtel's brief says sales leadership judges tools on **pipeline, conversion and growth**. So the dashboard reports exactly the three measures in the brief:
- **Qualified opportunities generated:** logged opportunities with score ≥ 60.
- **Sales conversion (proxy):** expected wins = sum of win probabilities. A real pilot would track actual stage changes in the CRM.
- **Customer value growth:** weighted new annual revenue, and **share of wallet** (our spend ÷ the customer's estimated ICT spend) now vs if won.

In [ ]:
import matplotlib.pyplot as plt

DASH_COLS = ["opp_id", "mode", "company", "solution_id", "score", "qualified", "est_value_sgd",
             "win_prob", "weighted_value_sgd", "sow_now", "sow_if_won", "timing", "status"]

def pipeline_dashboard():
    df = pd.DataFrame(PIPELINE)
    if df.empty:
        return pd.DataFrame(columns=DASH_COLS), "No opportunities logged yet."
    farm = df[df["mode"] == "Farm"]
    metrics = (f"**Qualified opportunities:** {df['qualified'].sum()} of {len(df)} logged (score ≥ {QUALIFIED_SCORE})  \n"
               f"**Conversion proxy:** {df['win_prob'].sum():.2f} expected wins (average win chance {df['win_prob'].mean():.0%})  \n"
               f"**Customer value growth:** S${df['weighted_value_sgd'].sum():,.0f} weighted new annual revenue")
    if not farm.empty:
        metrics += f"; share of wallet at farmed accounts {farm['sow_now'].mean():.0%} ➜ {farm['sow_if_won'].mean():.0%} if won"
    return df[DASH_COLS], metrics

dash_df, dash_metrics = pipeline_dashboard()
display(dash_df)
display(Markdown(dash_metrics))

if not dash_df.empty:
    ax = dash_df.plot.barh(x="company", y=["est_value_sgd", "weighted_value_sgd"], figsize=(7, 2.5),
                           title="Pipeline value per opportunity (S$ per year)")
    ax.invert_yaxis()
    plt.show()

## Step 10: Build the Demo UI with Gradio 🏆
Pick **Farm** (an existing customer) or **Hunt** (a market), get a brief, the ranked opportunities and an editable outreach draft. The **Approve** button logs to the pipeline and refreshes the dashboard.

⏱️ LLM requests per click: **Farm = 1**, **Hunt ≈ 5** (agent + brief). Approve and the dashboard use **no** LLM calls. On the free tier, give Hunt a minute.

In [ ]:
import gradio as gr

OPP_COLS = ["company", "solution_id", "solution", "score", "recency", "fit", "size", "renewal", "signal_ids"]

def scout(mode, account, market, industry):
    try:
        if mode.startswith("Farm"):
            company, agent_note = account, ""
            table = rank_opportunities(account)
        else:
            table = rank_prospects(market, industry)
            if table.empty:  # check before spending any LLM requests
                return f"No prospects found for {market} / {industry or 'any industry'}.", None, "", "", "", None
            agent_note = "**Agent shortlist**\n\n" + run_hunt_agent(market, industry) + "\n\n"
            company = top_from_log() or table.iloc[0]["company"]
        brief, problems, ev = make_brief(company)
    except Exception as e:
        return f"⚠️ Sorry, something went wrong: {e}", None, "", "", "", None
    checks = "\n".join(f"- ⚠️ {p}" for p in problems) or "✅ Every cited ID exists in the data."
    table = table[OPP_COLS].assign(signal_ids=table["signal_ids"].apply(", ".join))
    return (brief_markdown(brief, ev), table, brief.outreach_subject, brief.outreach_body,
            agent_note + "**ID check**\n\n" + checks, (brief, ev))

def approve(state, subject, body):
    if state is None:
        return "Run a scout first.", *pipeline_dashboard()
    brief, ev = state
    msg = log_opportunity(brief, ev, subject, body, approved_by="Account manager (demo)", approved=True)
    return msg, *pipeline_dashboard()

with gr.Blocks(title="AccountScout") as demo:
    gr.Markdown("# 📡 AccountScout: AI Sales Intelligence\nFarm existing accounts or hunt in a new market. "
                "Fictional data. Nothing is ever sent: you approve, it logs.")
    state = gr.State(None)
    with gr.Row():
        with gr.Column(scale=1):
            mode_in = gr.Radio(["Farm (existing customer)", "Hunt (new market)"], value="Farm (existing customer)", label="Mode")
            account_in = gr.Dropdown([a["company"] for a in CRM_ACCOUNTS], value=FARM_ACCOUNT, label="Account (Farm)")
            market_in = gr.Dropdown(["India", "Vietnam"], value="India", label="Market (Hunt)")
            industry_in = gr.Textbox(label="Industry filter (Hunt, optional)", placeholder="e.g. manufacturing")
            go = gr.Button("Scout", variant="primary")
            notes_out = gr.Markdown()
        with gr.Column(scale=2):
            brief_out = gr.Markdown(label="Account brief")
            opps_out = gr.Dataframe(label="Ranked opportunities (Python score)", wrap=True)
    with gr.Row():
        with gr.Column():
            subject_in = gr.Textbox(label="Outreach subject (edit me)")
            body_in = gr.Textbox(label="Outreach draft (edit me)", lines=8)
            approve_btn = gr.Button("✅ Approve and log to pipeline (does not send)", variant="secondary")
            approve_msg = gr.Markdown()
        with gr.Column():
            metrics_out = gr.Markdown(pipeline_dashboard()[1])
            pipeline_out = gr.Dataframe(pipeline_dashboard()[0], label="Pipeline", wrap=True)

    go.click(scout, inputs=[mode_in, account_in, market_in, industry_in],
             outputs=[brief_out, opps_out, subject_in, body_in, notes_out, state])
    approve_btn.click(approve, inputs=[state, subject_in, body_in], outputs=[approve_msg, pipeline_out, metrics_out])

# Launch it! (This will generate a public URL you can share with judges)
demo.launch(share=True, theme="soft")

## 🎯 What the Sponsor Wants After 36 Hours
From Singtel's brief, a strong outcome is:
- [ ] A **working prototype or compelling concept demo** that helps enterprise sales teams **identify qualified opportunities**.
- [ ] It **anticipates customer needs** (before the customer asks).
- [ ] It **recommends meaningful next actions**: right customer, right solution, right time.
- [ ] It shows **potential business impact** on at least one of: qualified opportunities generated, sales conversion, customer value growth (share of wallet, long-term revenue).
- [ ] It has a **practical approach to piloting and scaling**.

Copy this table into your pitch deck and fill it in:

| Question | Your answer |
|---|---|
| Who uses it, and is it Hunting, Farming or both? | |
| Which data sources does it join up? Who owns each one? | |
| Which measure does it move? Baseline ➜ expected (e.g. research time per account, qualified opps per AM per month) | |
| How does an insight become an action? (brief ➜ next action ➜ CRM opportunity) | |
| How do you stop wrong or made-up recommendations? | |
| Privacy: what personal data do you touch, and under which law (PDPA / DPDP)? | |
| Pilot: how many AMs, which market, how long, compared with what? | |
| Scale: what changes for a new country, segment or industry? | |

## ✅ Constraints Checklist
How this prototype measures up against the brief's constraints and scope:

- [x] **Clear business value:** every run ends in a scored opportunity in a pipeline, measured on the brief's three metrics.
- [x] **Translates insights into meaningful actions:** next best action, timing, a role to approach and an editable outreach draft.
- [x] **Applicability across markets, segments and industries:** market and industry are parameters; India and Vietnam prospects use the same code. Catalogue tags generalise to new industries.
- [x] **Data privacy (prototype level):** no personal data at all; roles only; cited, fictional sources. ⚠️ Not yet: access control, encryption, audit retention.
- [ ] **Integration with existing systems:** not yet. Mock lists stand in for the CRM. Pilot step: read-only connection to a CRM sandbox (e.g. Salesforce), then write approved opportunities back as CRM records.
- [ ] **Sustainable revenue growth / stronger relationships:** only estimated (share of wallet, weighted value). Proving it needs a pilot with real win/loss data.
- [ ] **Scalable and commercially viable:** cost per brief is 1-5 LLM requests; you still need to show the cost per user per month and who pays (enterprise sales and BD).
- [x] **In scope:** Hunting, Farming, customer and market intelligence, proactive next actions.
- [x] **Not out of scope:** not consumer-facing, not network operations, not billing or fulfilment, not pricing-only, and not a standalone chatbot.

## 🚀 Make It Your Own: Hackathon Extensions
Pick one or two. Don't try all of them.
- **Real signals:** pull public company news from an RSS feed or a news API, or use Gemini's Google Search grounding, and keep the source URL as the signal ID. Keep the mock list as a fallback.
- **Farm with the agent:** give the agent the CRM tools and let it compare all accounts to find *which* customer to call this week.
- **Learn the weights:** log won/lost outcomes and fit the score weights with a simple logistic regression.
- **Account plan co-creation:** turn the brief into a joint workshop agenda with the customer, moving from transactional selling to co-creating value.
- **Relationship map:** list the *roles* involved in a deal (economic buyer, technical buyer, user) and which ones we already have contact with in the CRM.
- **Local language outreach:** draft in Hindi or Tamil for the India market, with the account manager approving the translation.
- **Same pattern, different seller:** a bank's SME relationship managers, or a logistics company's key-account team.

## 🎤 Pitching to Judges
- ⚠️ **The brief rules out generic chatbots and standalone productivity tools** (meeting summaries, admin automation) unless they clearly link to acquisition, conversion or value growth. Don't pitch "a chatbot for sales". Pitch "**more qualified pipeline**".
- Lead with a **number tied to the three measures**: *"An account manager researches 10 accounts in 10 minutes instead of a day, and gets 3 qualified, cited opportunities with a next action."*
- Show **both journeys**: Farm (Kestrel's renewal and Pune plant) and Hunt (India prospects the team has never met).
- Show the **score breakdown**. Sales leaders trust a number they can explain.
- Show the **guardrails**: cited IDs checked in Python, roles not names, nothing sent without approval.
- End with the **pilot**: e.g. 10 account managers, one market (India), 8 weeks, compared with a control group, tracked in the existing CRM.

## 🛡️ Responsible AI Checklist
- [ ] **PDPA (Singapore) and India's DPDP Act:** collect only what you need, for a clear business purpose; personal data of contacts needs a lawful basis (e.g. consent).
- [ ] **No scraping of personal data** from LinkedIn or other professional networks. It breaks their terms and privacy law. Suggest roles; let the account manager find people through legitimate channels.
- [ ] **No fabricated people or facts:** every signal, solution and evidence ID is checked against real data (`check_brief`).
- [ ] **Cite sources:** every buying signal shows its source and date, so the account manager can verify it before using it.
- [ ] **Client confidentiality across accounts:** never use one customer's CRM data, tickets or deals in another customer's brief. Peer events must come from public news.
- [ ] **A human approves every outreach**, and nothing is sent automatically. Log who approved what (`approved_by`).
- [ ] **Fairness of the score:** check that smaller firms and less-covered regions aren't always ranked last just because there is less news about them.
- [ ] **Integration with least privilege:** start with read-only CRM access (e.g. a Salesforce sandbox) in the pilot; only approved records are written back.